
# Enriquecimiento de la tabla de inferencias con etiquetas reales

**Autor**: Juan Carlos Alfaro Jiménez

Esta libreta implementa el cierre del ciclo de monitorización del modelo en producción. Cada vez que el *endpoint* `fraud_lr_pipeline` recibe una petición, `Databricks Model Serving` registra automáticamente la *request* (atributos de la transacción) y la *response* (predicción) en la tabla de inferencias `fraud_lr_pipeline_payload`. En ese momento, la columna `is_fraud` es desconocida: la transacción acaba de ocurrir y aún no se sabe si es fraude.

Los fraudes se confirman con retraso. Cuando esa confirmación llega, se almacena en `silver_fraud_events` con la fecha en que la etiqueta está disponible (`label_available_date`). Esta libreta cruza ambas fuentes por `transaction_id` y propaga las etiquetas confirmadas a `gold_fraud_inference_enriched` mediante un `MERGE`, rellenando la columna `is_fraud` donde antes era `null`.

`Databricks Lakehouse Monitoring` opera sobre `gold_fraud_inference_enriched`. Solo las filas donde `is_fraud` no es `null` participan en el cálculo de métricas de rendimiento. Las filas pendientes de confirmación siguen participando en el análisis de *data drift* de características, para el que no se necesita la etiqueta.

Esta libreta se ejecuta como tarea `Update_Inference_Labels` dentro de `Credit Card Fraud Feature Pipeline`, que corre cada hora. En cada ejecución propaga únicamente las etiquetas de `silver_fraud_events` que todavía no tienen etiqueta confirmada en `gold_fraud_inference_enriched`, garantizando que la operación sea idempotente y compatible con datos históricos de cualquier periodo.

> El parseo de `transaction_id` depende de su posición dentro del *array* `data` de la *request*. El índice `transaction_id_idx = 0` asume que es la primera columna enviada al *endpoint*. Tras el primer despliegue, se debe verificar el orden real consultando la tabla de inferencias y ajustar el índice si fuera necesario.

## 1. Importaciones y configuración

In [0]:
exec(open("07_Utils.py").read(), globals())

In [0]:
from delta.tables import DeltaTable

from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    LongType,
    StringType,
    StructField,
    StructType
)

In [0]:
notebook_path_raw = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook = Path(notebook_path_raw).name

inference_payload_table = f"{catalog}.{database}.fraud_lr_pipeline_payload"
inference_enriched_table = f"{catalog}.{database}.gold_fraud_inference_enriched"
fraud_labels_table = f"{catalog}.{database}.silver_fraud_events"

# Index of transaction_id within the request columns array.
# Verify against the actual endpoint schema after the first invocation
# by inspecting a row in the inference payload table.
transaction_id_idx = 0

print(f"Project: {project}, team: {team}, environment: {environment}")
print(f"Notebook: {notebook}, user: {current_user}")
print(f"Inference payload table: {inference_payload_table}")
print(f"Inference enriched table: {inference_enriched_table}")
print(f"Fraud labels table: {fraud_labels_table}")


## 2. Lectura y parseo de la tabla de inferencias

La tabla de inferencias almacena cada petición y respuesta como `.json` en las columnas `request` y `response`. Se desempaquetan con `from_json` de `PySpark`, definiendo explícitamente el esquema esperado para garantizar tipos correctos y fallos explícitos ante cambios de contrato.

La *request* sigue el formato `dataframe_split` de `MLflow`: un objeto `.json` con claves `columns` y `data`. Se extrae `transaction_id` por posición dentro del *array* `data`, dado que el índice de columna es fijo y conocido en el momento del despliegue del *endpoint*. La *response* contiene el *array* `predictions` con la clase predicha por el modelo.

In [0]:
request_schema = StructType([
    StructField("columns", ArrayType(StringType())),
    StructField("data", ArrayType(ArrayType(StringType())))
])

response_schema = StructType([
    StructField("predictions", ArrayType(LongType()))
])

predictions_df = (
    spark.table(inference_payload_table)
         .withColumn("req", F.from_json(F.col("request"), request_schema))
         .withColumn("resp", F.from_json(F.col("response"), response_schema))
         .withColumn("transaction_id", F.col("req.data")[0][transaction_id_idx])
         .withColumn("prediction", F.col("resp.predictions")[0])
         .withColumn("request_timestamp", (F.col("timestamp_ms") / 1000).cast("timestamp"))
         .select(
             "transaction_id",
             "prediction",
             "request_timestamp"
         )
)

n_total = predictions_df.count()
n_fraud = predictions_df.filter(F.col("prediction") == 1).count()
n_legit = predictions_df.filter(F.col("prediction") == 0).count()

print(f"Parsed predictions: {n_total:,} total ({n_fraud:,} fraud, {n_legit:,} legit)")


## 3. Lectura de las etiquetas pendientes de propagar

Se leen de `silver_fraud_events` únicamente las etiquetas cuyo `transaction_id` no tiene todavía una etiqueta confirmada en `gold_fraud_inference_enriched`. Este enfoque es idempotente: si el trabajo falla y se relanza, no se pierden las etiquetas ni se introducen duplicados. Es además compatible con datos históricos de cualquier periodo, ya que no depende de la fecha actual sino del estado real de la tabla enriquecida.

In [0]:
already_labelled_df = (
    DeltaTable.forName(spark, inference_enriched_table)
              .toDF()
              .filter(F.col(label_column).isNotNull())
              .select("transaction_id")
)

pending_labels_df = (
    spark.table(fraud_labels_table)
         .join(already_labelled_df, on = "transaction_id", how = "left_anti")
         .select("transaction_id", label_column)
)

n_pending_total = pending_labels_df.count()
n_pending_fraud = pending_labels_df.filter(F.col(label_column) == 1).count()
n_pending_legit = pending_labels_df.filter(F.col(label_column) == 0).count()

print(f"Labels pending propagation: {n_pending_total:,} total ({n_pending_fraud:,} fraud, {n_pending_legit:,} legit)")


## 4. Propagación de etiquetas a la tabla enriquecida

Se une la tabla de predicciones parseadas con las etiquetas pendientes de propagar y se escribe el resultado en `gold_fraud_inference_enriched` mediante la `Delta API` de `PySpark` (`DeltaTable.merge`):

* Si la transacción ya existe en la tabla enriquecida y la etiqueta acaba de confirmarse, se actualiza `is_fraud`.
* Si la transacción aún no existe en la tabla enriquecida, se inserta con la etiqueta disponible si ya está confirmada, o con `is_fraud` nulo si todavía no lo está.
* Las filas sin etiqueta confirmada permanecen con `is_fraud` nulo y participan en el análisis de *data drift* de características hasta que la etiqueta llegue en una ejecución posterior.

In [0]:
enriched_update_df = predictions_df.join(
    pending_labels_df,
    on = "transaction_id",
    how = "left"
)

(
    DeltaTable.forName(spark, inference_enriched_table)
              .alias("target")
              .merge(
                  enriched_update_df.alias("source"),
                  "target.transaction_id = source.transaction_id"
              )
              .whenMatchedUpdate(
                  condition = f"source.{label_column} IS NOT NULL",
                  set = {f"target.{label_column}": f"source.{label_column}"}
              )
              .whenNotMatchedInsert(
                  values = {
                      "transaction_id": "source.transaction_id",
                      "prediction": "source.prediction",
                      "request_timestamp": "source.request_timestamp",
                      label_column: f"source.{label_column}"
                  }
              )
              .execute()
)

print(f"MERGE completed into {inference_enriched_table}")


## 5. Conclusiones y siguientes pasos

### ¿Qué hace esta libreta?

Esta libreta implementa el paso de cierre del ciclo de monitorización:

1. **Parseo de la tabla de inferencias**: Desempaqueta el `.json` de *requests* y *responses* del *endpoint* usando `from_json` de `PySpark` con esquema explícito, extrayendo `transaction_id`, `prediction` y `request_timestamp` como columnas tipadas.
2. **Lectura de etiquetas pendientes**: Lee únicamente las etiquetas de `silver_fraud_events` que todavía no tienen etiqueta confirmada en `gold_fraud_inference_enriched`, garantizando que la operación sea idempotente y compatible con datos históricos de cualquier periodo.
3. **`MERGE` incremental con la `Delta API`**: Propaga las etiquetas a `gold_fraud_inference_enriched` sin reescribir el histórico. Las transacciones sin etiqueta permanecen con `is_fraud` nulo hasta que se confirmen en una ejecución posterior.

### ¿Por qué es necesaria?

`Databricks Lakehouse Monitoring` necesita las etiquetas reales para calcular métricas de rendimiento en producción. Sin este paso, el monitor solo puede calcular *data drift* de características pero no detectar degradación real del modelo. Con las etiquetas propagadas, el monitor compara el rendimiento real en producción contra el *baseline* de `gold_fraud_test_baseline`.

Una vez que la tabla enriquecida empiece a recibir datos, se configura manualmente una alerta sobre las tablas de métricas del monitor. Cuando la degradación del rendimiento supere el umbral definido en esa alerta, se dispara automáticamente el trabajo `Credit Card Fraud Retraining Pipeline`.

### ¿Cuándo se ejecuta?

Como tarea `Update_Inference_Labels` en el trabajo `Credit Card Fraud Feature Pipeline`, que corre cada hora. Es la última tarea del *pipeline*, después de `Publish_to_Online_Store`, para garantizar que las etiquetas generadas en el ciclo actual ya están disponibles en `silver_fraud_events`.